## Course map

| Notebook | Main focus |
|---|---|
| Beginner | DX-RT role, device checks, DXNN inspection, and CLI inference |
| Intermediate | Python and C++ APIs, synchronous/asynchronous execution, batch, and buffers |
| Advanced | Resource binding, profiling, monitoring, multi-input/memory loading, and release validation |

Complete the first two notebooks before tuning runtime resources. Optimization without a verified tensor contract and baseline is difficult to interpret.


# DX-RT Tutorial 3: Advanced

This notebook turns a working inference call into an observable and reproducible runtime experiment.

## Learning objectives

By the end of this tutorial, you will be able to:

- configure device selection, NPU core binding, ORT use, and buffer count,
- tune buffer count with a controlled benchmark,
- generate and visualize a DX-RT profiler trace,
- read per-job H2D, NPU, D2H, format-handler, and CPU-task metrics,
- use Coefficient of Variation to detect unstable timing,
- query device memory, utilization, and thermal status,
- load a DXNN model from an in-memory buffer,
- prepare named inputs for a multi-input model,
- register a runtime event handler,
- explain the stable C ABI, header-only C++ wrapper, and runtime IPC boundary,
- explain, build, enable, and A/B test NFH and CPU-op acceleration through C++, Python, and `dxrun`, and
- create a compact release-validation record.

Every generated trace, report, source file, and result stays under <code>&lt;dx-tutorials&gt;/notebooks/T06-DX-Runtime/workspace</code>.


## 1. Initialize the tutorial workspace

This notebook expects the matching <code>dx_engine</code> wheel in the T06-local environment created by the Intermediate tutorial. It also links a second model when available so that <code>dxbenchmark</code> can demonstrate a multi-model report.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shlex
import sys
import time

import numpy as np
from IPython.display import Image, display

root_path = os.environ.get("ROOT_PATH")
if not root_path:
    raise EnvironmentError("ROOT_PATH is not set. Start JupyterLab with ./run-jupyter-lab.sh")

%run "$root_path/tutorial_paths.py"

T06_DIR = TUTORIAL_ROOT / "notebooks" / "T06-DX-Runtime"
WORK_DIR = T06_DIR / "workspace"
MODEL_DIR = WORK_DIR / "models"
REPORT_DIR = WORK_DIR / "reports"
PROFILER_DIR = WORK_DIR / "profiler"
DXRT_PYTHON_ENV = WORK_DIR / ".venv-dxrt"
DXRT_PYTHON = DXRT_PYTHON_ENV / "bin" / "python"
DXRT_SITE_PACKAGES = (
    DXRT_PYTHON_ENV
    / "lib"
    / f"python{sys.version_info.major}.{sys.version_info.minor}"
    / "site-packages"
)

for path in (WORK_DIR, MODEL_DIR, REPORT_DIR, PROFILER_DIR):
    path.mkdir(parents=True, exist_ok=True)

model_sources = [
    DX_WORKSPACE_DIR / "res" / "models" / "resnet50_224x224.dxnn",
    DX_WORKSPACE_DIR / "res" / "models" / "mobilenetv2_224x224.dxnn",
]
if not model_sources[0].is_file():
    command = (
        f"cd {shlex.quote(str(DX_APP_DIR))} && "
        "bash setup.sh --models resnet50 mobilenetv2 --no-force"
    )
    raise FileNotFoundError(
        f"Required model was not found: {model_sources[0]}\n"
        f"Run this command in a terminal, then rerun this cell:\n{command}"
    )

for source in model_sources:
    if source.is_file():
        link = MODEL_DIR / source.name
        if not link.exists():
            link.symlink_to(source)
    else:
        print(f"Optional comparison model is missing: {source.name}")

MODEL_PATH = MODEL_DIR / model_sources[0].name
if not DXRT_SITE_PACKAGES.is_dir():
    raise FileNotFoundError(
        "The T06-local DX-RT Python environment is missing. "
        "Complete Intermediate Section 2 first."
    )
sys.path.insert(0, str(DXRT_SITE_PACKAGES))
os.chdir(WORK_DIR)

print(f"Workspace: {WORK_DIR}")
print("Models:")
for path in sorted(MODEL_DIR.glob("*.dxnn")):
    print(f"  {path.name} -> {path.resolve()}")


In [ ]:
try:
    from dx_engine import (
        Configuration,
        DeviceStatus,
        InferenceEngine,
        InferenceOption,
        Profiler,
        RuntimeEventDispatcher,
        __version__ as dx_engine_version,
    )
except ImportError as error:
    raise ImportError(
        "The Jupyter kernel cannot import dx_engine. Complete Intermediate Section 2 first.\n"
        f"Expected local Python environment: {DXRT_PYTHON_ENV}"
    ) from error

print(f"dx_engine version: {dx_engine_version}")


## 2. Runtime resource control

<code>InferenceOption</code> controls where and how one engine instance runs.

| Option | Default meaning | Change it when |
|---|---|---|
| <code>devices</code> | Empty list uses available devices | Isolating a device or distributing engines |
| <code>bound_option</code> | <code>NPU_ALL</code> | Reserving one core or a core pair |
| <code>use_ort</code> | Build-dependent | The model contains CPU tasks |
| <code>buffer_count</code> | Runtime default, normally 6 | Measured queueing or memory behavior requires tuning |

A device selects a physical accelerator. A bound option selects cores inside each selected device. Do not confuse these two levels.


In [ ]:
device_count = DeviceStatus.get_device_count()
print(f"Detected devices: {device_count}")

option = InferenceOption()
option.devices = [0]
option.bound_option = InferenceOption.BOUND_OPTION.NPU_ALL
option.use_ort = False
option.buffer_count = 6
print(option)


### 2.1 Binding safety

If existing engine instances already reserve individual NPU cores, creating another engine with <code>NPU_ALL</code> can block until all required cores are released. Keep resource ownership explicit in multi-threaded and multi-process services.

| Goal | Typical starting point |
|---|---|
| Maximum throughput for one engine | One or more devices, <code>NPU_ALL</code> |
| Isolate two independent pipelines | Assign different devices or non-overlapping core bindings |
| Reproduce a single-core latency test | One device and one core |
| Share hardware across processes | Define ownership before creating engines |

Binding is a deployment decision. The notebook keeps its executed labs on device 0 with <code>NPU_ALL</code>.


## 3. Tune buffer count with a controlled experiment

<img src="assets/dx-rt-observability.svg" style="max-width: 1100px; width: 100%;" alt="DX-RT observability and tuning flow">

The four commands below differ only in <code>--buffer-count</code>. They use the same model, warm-up count, duration, and execution mode. Compare throughput and latency; do not select the largest count automatically.

Equivalent terminal pattern:

~~~bash
cd <T06-DX-Runtime>/workspace
dxrun -m models/resnet50_224x224.dxnn \
      --benchmark --time 3 --warmup-runs 5 --buffer-count N
~~~


In [ ]:
!cd "{WORK_DIR}" && dxrun -m "models/{MODEL_PATH.name}" --benchmark --time 3 --warmup-runs 5 --buffer-count 2
!cd "{WORK_DIR}" && dxrun -m "models/{MODEL_PATH.name}" --benchmark --time 3 --warmup-runs 5 --buffer-count 4
!cd "{WORK_DIR}" && dxrun -m "models/{MODEL_PATH.name}" --benchmark --time 3 --warmup-runs 5 --buffer-count 6
!cd "{WORK_DIR}" && dxrun -m "models/{MODEL_PATH.name}" --benchmark --time 3 --warmup-runs 5 --buffer-count 8


Choose the smallest value that reaches the required stable throughput without violating memory or latency limits. Repeat the experiment under the real application load because preprocessing, post-processing, and other processes change queue pressure.


## 4. Profile the runtime timeline

The profiler separates time into stages such as input formatting, host-to-device transfer, NPU compute, device-to-host transfer, output formatting, and CPU tasks.

The following command creates <code>profiler.json</code> inside the T06 profiler directory:

~~~bash
cd <T06-DX-Runtime>/workspace/profiler
dxrun -m ../models/resnet50_224x224.dxnn \
      --benchmark --time 5 --warmup-runs 5 --profiler
~~~


In [ ]:
!cd "{PROFILER_DIR}" && dxrun     -m "../models/{MODEL_PATH.name}"     --benchmark     --time 5     --warmup-runs 5     --profiler


### 4.1 Convert the trace to images

The SDK plotting command is shown directly below. <code>--auto-select</code> focuses on a stable center region, and the output remains under T06.

Equivalent terminal command:

~~~bash
python <DX_RT_DIR>/tool/profiler/plot.py \
  --input <T06>/workspace/profiler/profiler.json \
  --output <T06>/workspace/profiler/profiler.png \
  --auto-select
~~~


In [ ]:
PROFILER_JSON = PROFILER_DIR / "profiler.json"
PROFILER_PLOT = DX_RT_DIR / "tool" / "profiler" / "plot.py"
PROFILER_IMAGE = PROFILER_DIR / "profiler.png"

if not PROFILER_JSON.is_file():
    raise FileNotFoundError(f"Profiler output was not created: {PROFILER_JSON}")

!"{sys.executable}" "{PROFILER_PLOT}"     --input "{PROFILER_JSON}"     --output "{PROFILER_IMAGE}"     --auto-select


In [ ]:
profiler_images = sorted(PROFILER_DIR.glob("profiler*.png"))
if not profiler_images:
    raise FileNotFoundError(f"No profiler image was created under {PROFILER_DIR}")

for image_path in profiler_images:
    print(image_path.name)
    display(Image(filename=str(image_path)))


### 4.2 Interpret profiler events

| Event | What it measures | First question |
|---|---|---|
| Buffer Wait | Waiting for a free inference buffer | Is queue depth or CPU pressure too high? |
| NPU Input Format Handler | Padding and layout conversion | Is format conversion a significant fraction? |
| PCIe Write / H2D | Host-to-device transfer | Is input size or transfer the limit? |
| NPU Core | NPU computation | Does the model dominate total time? |
| PCIe Read / D2H | Device-to-host transfer | Are outputs unusually large? |
| NPU Output Format Handler | Output slicing and layout conversion | Is conversion the bottleneck? |
| CPU Task Queue Wait | Waiting for CPU execution | Is the CPU pipeline saturated? |
| cpu_N | CPU operator execution | Are compute-heavy CPU operators dominant? |

The total NPU task covers format handling, transfer, compute, and output handling. It is broader than NPU-core compute alone.


## 5. Read per-job metrics through the Python API

DX-RT v3.4 adds <code>get_job_metrics(job_id)</code>. Call it immediately after <code>wait(job_id)</code>. The older performance-data methods are deprecated.

The code below enables profiling, runs 20 asynchronous jobs, and prints the final job's valid metrics. The repeated jobs also provide enough samples for the CoV table in Section 5.1.


In [ ]:
config = Configuration()
config.set_enable(Configuration.ITEM.PROFILER, True)
config.set_attribute(
    Configuration.ITEM.PROFILER,
    Configuration.ATTRIBUTE.PROFILER_SHOW_DATA,
    "OFF",
)
config.set_attribute(
    Configuration.ITEM.PROFILER,
    Configuration.ATTRIBUTE.PROFILER_SAVE_DATA,
    "OFF",
)

profiler = Profiler.get_instance()
profiler.clear()

option = InferenceOption()
option.devices = [0]
option.bound_option = InferenceOption.BOUND_OPTION.NPU_ALL
option.buffer_count = 4
PROFILED_JOBS = 20

with InferenceEngine(str(MODEL_PATH), option) as ie:
    info = ie.get_input_tensors_info()[0]
    input_tensor = np.empty(info["shape"], dtype=info["dtype"])
    input_tensor.fill(0)

    input_tensor = np.ascontiguousarray(input_tensor)
    for _ in range(PROFILED_JOBS):
        job_id = ie.run_async([input_tensor])
        outputs = ie.wait(job_id)
        metrics = profiler.get_job_metrics(job_id)

print(f"Profiled jobs: {PROFILED_JOBS}")
print(f"Final job ID : {job_id}")
print(f"Valid  : {metrics.valid}")
for task in metrics.tasks:
    print(f"Task   : {task.task_name}")
    for device_id, device in task.devices.items():
        print(
            f"  device={device_id}, input_format={device.input_format_us:.3f} us, "
            f"H2D={device.h2d_us:.3f} us, NPU={device.inference_core_all_us:.3f} us, "
            f"D2H={device.d2h_us:.3f} us, output_format={device.output_format_us:.3f} us, "
            f"total={device.total_us:.3f} us"
        )
    if task.cpu_task_us > 0:
        print(f"  CPU task={task.cpu_task_us:.3f} us")


### 5.1 Stability with Coefficient of Variation

Coefficient of Variation is standard deviation divided by mean:

~~~text
CoV (%) = standard deviation / mean × 100
~~~

It compares relative jitter across stages with different average durations. Lower is more stable, but there is no universal release threshold. Define one from the product's latency budget and operating conditions.

| Result | Interpretation |
|---|---|
| Low mean, high CoV | Usually fast but occasionally unstable |
| High mean, low CoV | Predictably slow |
| High p99, moderate mean | Tail latency needs investigation |
| Stable NPU, unstable host time | Inspect host queues, CPU load, and preprocessing |


In [ ]:
profiler.show()


## 6. Monitor device health

The monitoring service updates shared status approximately once per second. Polling faster usually returns the same sample.

The next cell records two snapshots. Treat <code>is_valid() == False</code> as stale or unavailable monitoring data.


In [ ]:
def device_snapshot(device_id: int) -> dict:
    status = DeviceStatus.get_current_status(device_id)
    return {
        "device": status.get_id(),
        "valid": status.is_valid(),
        "driver": status.get_driver_version(),
        "memory_used_bytes": status.get_memory_used(),
        "memory_free_bytes": status.get_memory_free(),
        "temperature_c": [status.get_temperature(core) for core in range(3)],
        "utilization_percent": [
            status.get_core_utilization(core) for core in range(3)
        ],
        "clock_mhz": [status.get_npu_clock(core) for core in range(3)],
    }

snapshots = []
for _ in range(2):
    snapshots.append([device_snapshot(i) for i in range(device_count)])
    time.sleep(1.0)

print(json.dumps(snapshots, indent=2))


For continuous inspection, run <code>dxtop</code> in a separate terminal while the application runs. Collect temperature, utilization, memory, and throttling evidence together with latency and throughput; a short cool-system benchmark can hide production thermal behavior.


## 7. Load a model from memory

Memory loading is useful when the model comes from encrypted storage, a package, a network service, or another managed data source. The NumPy array must be C-contiguous and must remain alive for the engine's lifetime.


In [ ]:
model_buffer = np.fromfile(MODEL_PATH, dtype=np.uint8)
model_buffer = np.ascontiguousarray(model_buffer)

with InferenceEngine.from_buffer(model_buffer) as ie:
    info = ie.get_input_tensors_info()[0]
    input_tensor = np.empty(info["shape"], dtype=info["dtype"])
    input_tensor.fill(0)
    memory_outputs = ie.run([np.ascontiguousarray(input_tensor)])

print(f"Model buffer bytes : {model_buffer.nbytes}")
print(f"Output tensors     : {len(memory_outputs)}")


## 8. Multi-input model contract

For a multi-input model, the safest interface is a dictionary keyed by exact tensor names:

~~~python
inputs = {
    "left_image": left_tensor,
    "right_image": right_tensor,
}
outputs = engine.run_multi_input(inputs)
~~~

DX-RT also supports ordered lists and one concatenated buffer, but named inputs reduce ordering mistakes.

This lab reuses the two-input DXNN created by T05 Advanced Section 8 when it exists. It does not compile a model or create files outside T06.


In [ ]:
T05_MULTI_INPUT_DIR = (
    TUTORIAL_ROOT
    / "notebooks"
    / "T05-DX-Compiler"
    / "outputs"
    / "stereo_fusion_python_api"
)
multi_input_candidates = sorted(T05_MULTI_INPUT_DIR.glob("*.dxnn"))
MULTI_INPUT_MODEL = None

if multi_input_candidates:
    source = max(multi_input_candidates, key=lambda path: path.stat().st_mtime_ns)
    MULTI_INPUT_MODEL = MODEL_DIR / source.name
    if not MULTI_INPUT_MODEL.exists():
        MULTI_INPUT_MODEL.symlink_to(source)
    print(f"Multi-input model: {MULTI_INPUT_MODEL} -> {source}")
else:
    print(
        "Multi-input execution lab skipped. Complete T05 Advanced Section 8 "
        "to create the stereo-fusion DXNN, then rerun this cell."
    )


In [ ]:
if MULTI_INPUT_MODEL is not None:
    with InferenceEngine(str(MULTI_INPUT_MODEL)) as ie:
        if not ie.is_multi_input_model():
            raise ValueError(f"Expected a multi-input model: {MULTI_INPUT_MODEL}")

        named_inputs = {}
        for info in ie.get_input_tensors_info():
            tensor = np.empty(info["shape"], dtype=info["dtype"])
            tensor.fill(0)
            named_inputs[info["name"]] = np.ascontiguousarray(tensor)

        multi_outputs = ie.run_multi_input(named_inputs)
        print("Input names :", list(named_inputs))
        print("Output count:", len(multi_outputs))
else:
    print("Skipped: no multi-input DXNN is available.")


## 9. Runtime events and service integration

<code>RuntimeEventDispatcher</code> centralizes device warnings, errors, recovery notices, memory events, and throttling events. A product handler should be fast, thread-safe, and should forward structured events to the application's logging or health system.

The next cell registers a handler and dispatches one **synthetic tutorial event** to verify the route. It does not simulate a real hardware fault.


In [ ]:
received_events = []

def tutorial_event_handler(level, event_type, code, message, timestamp):
    event = {
        "timestamp": timestamp,
        "level": int(level),
        "type": int(event_type),
        "code": int(code),
        "message": message,
    }
    received_events.append(event)
    print(json.dumps(event, indent=2))

dispatcher = RuntimeEventDispatcher()
dispatcher.set_current_level(RuntimeEventDispatcher.LEVEL.INFO)
dispatcher.register_event_handler(tutorial_event_handler)
dispatcher.dispatch_event(
    RuntimeEventDispatcher.LEVEL.INFO,
    RuntimeEventDispatcher.TYPE.DEVICE_STATUS,
    RuntimeEventDispatcher.CODE.DEVICE_EVENT,
    "Synthetic T06 event: handler route verified.",
)


### 9.1 ABI and IPC boundaries

DX-RT v3.4 separates public integration from internal implementation:

| Layer | Purpose | Product guidance |
|---|---|---|
| Stable C ABI, <code>dxrt_c_api.h</code> | Versioned C symbols from <code>libdxrt.so</code> | Use for language bindings and binary distribution |
| Header-only C++14 wrapper, <code>dxrt_cxx_api.h</code> | Modern C++ interface over the C ABI | Recommended for new C++ applications |
| Legacy bridge headers | Existing <code>dxrt_api.h</code> source compatibility | Keep old products building; migrate deliberately |
| Shared-memory IPC and runtime service | Efficient process-to-runtime communication | Treat as an internal transport, not an application tensor API |

Do not depend on hidden C++ symbols from <code>libdxrt.so</code>. Public C/C++ headers are the supported integration boundary.


## 10. Optional CPU-side acceleration

These features accelerate host-side runtime work. They do not change the DXNN graph or make the NPU compute layers faster. Enable them only after the profiler identifies the matching bottleneck.

| Feature | Work accelerated | x86_64 implementation | aarch64 implementation | Useful when |
|---|---|---|---|---|
| `NPU_FORMAT_CONVERSION_ACCELERATION` | NPU Format Handler (NFH): transpose, padding, slicing, and device-layout conversion around NPU tasks | Intel IPP | ARM NEON/ASIMD | Input or output format-handler time is significant |
| `CPU_OP_ACCELERATION` | ONNX Runtime CPU operators in CPU fallback subgraphs | OpenVINO Execution Provider | XNNPACK Execution Provider | ORT CPU time is dominated by compute-heavy operators such as Conv or MatMul |

NFH acceleration does **not** replace application preprocessing. Resize, color conversion, normalization, and other model-specific work still follow the model contract. CPU-op acceleration does **not** move CPU operators to the NPU; it selects a more optimized CPU execution provider.

### 10.1 Two gates: build support and runtime opt-in

Both gates must be open:

| Gate | Purpose | Default |
|---|---|---|
| Build-time CMake option | Compiles the feature and its platform library into DX-RT | `OFF` |
| Runtime setting | Enables the compiled feature for the process | `OFF` |

If build support is absent, the C++ enum, Python enum, and corresponding `dxrun` option do not exist. Runtime opt-in alone cannot add the missing implementation.

### 10.2 Build DX-RT with acceleration support

In the DX-RT source tree, edit `<DX_RT_DIR>/cmake/dxrt.cfg.cmake` and change only these two options:

~~~cmake
option(USE_NPU_FORMAT_CONVERSION_ACCELERATION
       "Accelerate NPU data format conversion (transpose/padding)" ON)
option(USE_CPU_OP_ACCELERATION
       "Accelerate CPU-side ONNX operations" ON)
~~~

Then perform a clean build so CMake re-detects the required libraries:

~~~bash
cd <DX_RT_DIR>
./build.sh --clean
~~~

`CPU_OP_ACCELERATION` also requires an ORT-enabled DX-RT build because it selects an optimized ONNX Runtime execution provider. A clean build may download or install platform dependencies. The build disables a requested feature when its required library or platform requirement is unavailable. Check the CMake output instead of assuming that `ON` succeeded. After installing a new runtime, reinstall the matching `dx_engine` wheel used by the application.


### 10.3 Verify that the installed runtime exposes the features

Check the Python wheel and the system CLI separately. They can have different build capabilities even when they report the same version. The following cell is read-only. A consistent deployment should expose the required feature in every interface that the product uses.


In [ ]:
feature_items = {
    "NPU format conversion": "NFH_ACCELERATION",
    "CPU operator": "CPU_OP_ACCELERATION",
}
for label, item_name in feature_items.items():
    state = "available" if hasattr(Configuration.ITEM, item_name) else "not compiled in"
    print(f"{label:24}: {state}")

print("\nMatching system dxrun options:")
!dxrun --help | grep -E -- '--accel-(nfh|cpu)' || echo 'No acceleration options are compiled into this dxrun.'


### 10.4 Enable the features in an application

Configure acceleration **before** constructing the first `InferenceEngine`. Keep ORT enabled when the DXNN contains CPU tasks.

**C++**

~~~cpp
#include <dxrt/dxrt_cxx_api.h>

int main()
{
    auto& config = dxrt::Configuration::GetInstance();

#ifdef DXRT_NFH_ACCELERATION_AVAILABLE
    config.SetEnable(
        dxrt::Configuration::ITEM::NFH_ACCELERATION, true);
#endif

#ifdef DXRT_CPU_OP_ACCELERATION_AVAILABLE
    config.SetEnable(
        dxrt::Configuration::ITEM::CPU_OP_ACCELERATION, true);
#endif

    dxrt::InferenceOption option;
    option.useORT = true;  // Required only when the graph has CPU tasks.
    dxrt::InferenceEngine engine("model.dxnn", &option);
    // Prepare input and run inference.
}
~~~

The preprocessor guards keep the source buildable when an acceleration feature is not present in the installed DX-RT headers. A product can instead treat a missing feature as a configuration error.

**Python**

~~~python
from dx_engine import Configuration, InferenceEngine, InferenceOption

config = Configuration()
required_items = ("NFH_ACCELERATION", "CPU_OP_ACCELERATION")
missing = [name for name in required_items if not hasattr(Configuration.ITEM, name)]
if missing:
    raise RuntimeError(f"DX-RT was built without: {', '.join(missing)}")

config.set_enable(Configuration.ITEM.NFH_ACCELERATION, True)
config.set_enable(Configuration.ITEM.CPU_OP_ACCELERATION, True)

option = InferenceOption()
option.use_ort = True  # Required only when the graph has CPU tasks.
with InferenceEngine("model.dxnn", option) as engine:
    # Prepare input and run inference.
    pass
~~~

The Python wheel must match the newly installed runtime version and Python ABI. Otherwise the enum or native extension may not match the shared library.


### 10.5 Test with `dxrun`

First confirm that `dxrun --help` lists `--accel-nfh` and `--accel-cpu`. Then run an A/B comparison with a DXNN model that contains CPU tasks. Keep the model, ORT setting, duration, warm-up, buffer count, device binding, and system load identical.

~~~bash
MODEL=/path/to/model-with-cpu-tasks.dxnn

# 1. Baseline
dxrun -m "$MODEL" --use-ort --benchmark --time 10 --warmup-runs 10 --buffer-count 6

# 2. Accelerate only NPU format conversion
dxrun -m "$MODEL" --use-ort --benchmark --time 10 --warmup-runs 10 --buffer-count 6 --accel-nfh

# 3. Accelerate only ORT CPU operators
dxrun -m "$MODEL" --use-ort --benchmark --time 10 --warmup-runs 10 --buffer-count 6 --accel-cpu

# 4. Enable both features
dxrun -m "$MODEL" --use-ort --benchmark --time 10 --warmup-runs 10 --buffer-count 6 --accel-nfh --accel-cpu
~~~

Add `--profiler` to a shorter diagnostic run when you need stage-level evidence. Throughput alone cannot tell whether NFH, CPU operators, transfers, or NPU compute changed.

| Result | Interpretation |
|---|---|
| NFH time decreases | Format-conversion acceleration is working |
| CPU task time decreases | Optimized ORT execution provider is helping |
| FPS is unchanged | Another stage is the bottleneck or the accelerated work is too small |
| Latency or CPU usage becomes worse | Disable the feature for this workload and retain the baseline |

`CPU_OP_ACCELERATION` is useful mainly for compute-heavy CPU operations. Reshape, Transpose, and Concat are often memory-bound and may show little improvement. Neither feature guarantees a performance gain.

### 10.6 Related option: dynamic CPU threading

If the profiler shows CPU task queue pressure rather than expensive individual operators, test dynamic CPU threading separately:

~~~bash
export DXRT_DYNAMIC_CPU_THREAD=ON
dxrun -m "$MODEL" --use-ort --benchmark --time 10 --warmup-runs 10
~~~

This is another independent A/B experiment. It does not replace NFH or CPU-op acceleration.


## 11. Benchmark several models

<code>dxbenchmark</code> finds DXNN files in a directory and creates machine-readable and visual reports. The result path is inside T06.

Equivalent terminal command:

~~~bash
cd <T06>/workspace/reports/dxbenchmark
dxbenchmark --dir <T06>/workspace/models \
            --result-path <T06>/workspace/reports/dxbenchmark \
            --time 3 \
            --warmup 3 \
            --sort fps \
            --order desc
~~~


In [ ]:
DXBENCHMARK_DIR = REPORT_DIR / "dxbenchmark"
DXBENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

!cd "{DXBENCHMARK_DIR}" && dxbenchmark     --dir "{MODEL_DIR}"     --result-path "{DXBENCHMARK_DIR}"     --time 3     --warmup 3     --sort fps     --order desc


In [ ]:
print("Generated benchmark files:")
for path in sorted(DXBENCHMARK_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(WORK_DIR))


## 12. Create a release-validation record

<img src="assets/dx-rt-release-loop.svg" style="max-width: 1100px; width: 100%;" alt="Production runtime validation loop">

A runtime release record should connect the exact binary and environment to measured evidence. The next cell writes a compact JSON record inside T06. Add product accuracy, end-to-end latency, host CPU, memory, and thermal results before a real release decision.


In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

release_record = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model": {
        "path": str(MODEL_PATH),
        "resolved_path": str(MODEL_PATH.resolve()),
        "sha256": sha256(MODEL_PATH),
    },
    "software": {
        "dx_engine": dx_engine_version,
        "kernel_python": sys.version.split()[0],
    },
    "hardware": {
        "device_count": device_count,
        "latest_snapshot": snapshots[-1] if snapshots else [],
    },
    "tutorial_evidence": {
        "profiler_json": str(PROFILER_JSON),
        "dxbenchmark_directory": str(DXBENCHMARK_DIR),
    },
    "product_evidence_to_add": {
        "accuracy_metric": None,
        "end_to_end_latency_ms": None,
        "throughput_fps": None,
        "host_cpu_percent": None,
        "peak_memory_bytes": None,
        "thermal_soak_result": None,
    },
}

RELEASE_RECORD = REPORT_DIR / "release_validation.json"
RELEASE_RECORD.write_text(json.dumps(release_record, indent=2) + "\n", encoding="utf-8")
print(RELEASE_RECORD)
print(json.dumps(release_record, indent=2))


## 13. Summary

### 13.1 Production workflow completed

**Freeze the contract and workload**  
→ **Measure a baseline**  
→ **Profile each runtime stage**  
→ **Tune one resource at a time**  
→ **Monitor stability and device health**  
→ **Store reproducible release evidence**

<img src="assets/dx-rt-release-loop.svg" style="max-width: 1000px; width: 100%;" alt="DX-RT production validation loop">

### 13.2 Advanced control dashboard

| Decision | Evidence first | Control |
|---|---|---|
| Queue depth | Buffer Wait, latency, memory | <code>buffer_count</code> |
| Resource placement | Device/core utilization | <code>devices</code>, <code>bound_option</code> |
| CPU fallback | Task graph and CPU task time | <code>use_ort</code> |
| Format conversion | Input/output format-handler time | NFH acceleration if compiled |
| CPU operator cost | CPU task type and duration | CPU-op acceleration if compiled |
| CPU queue pressure | Queue wait and host CPU | Dynamic CPU threading |
| Service health | Runtime events and device snapshots | Event handler and monitoring policy |

### 13.3 Evidence produced

| Artifact | Location | Purpose |
|---|---|---|
| Profiler JSON | <code>workspace/profiler/profiler.json</code> | Raw event timeline |
| Profiler images | <code>workspace/profiler/profiler*.png</code> | Visual bottleneck inspection |
| Benchmark reports | <code>workspace/reports/dxbenchmark/</code> | Multi-model comparison |
| Release record | <code>workspace/reports/release_validation.json</code> | Traceable release checklist |

### 13.4 Completion checklist

- [x] Configured devices, core binding, ORT, and buffer count
- [x] Compared several buffer counts under one fixed workload
- [x] Generated and visualized a profiler trace
- [x] Read per-job stage metrics
- [x] Used CoV as a timing-stability signal
- [x] Queried memory, utilization, clock, and thermal status
- [x] Loaded a DXNN model from memory
- [x] Prepared a named multi-input inference path
- [x] Verified a runtime-event handler with a synthetic event
- [x] Distinguished acceleration build support from runtime opt-in
- [x] Mapped NFH and CPU-op acceleration to C++, Python, and `dxrun` controls
- [x] Created benchmark and release-record artifacts under T06
- [ ] Run the same tests with real preprocessing and post-processing
- [ ] Validate task accuracy with representative labeled data
- [ ] Run a thermal soak and tail-latency test on the deployment host

> **Remember:** optimize the stage that the profiler identifies. A faster isolated NPU result is not a release result until accuracy, end-to-end latency, CPU load, memory, stability, and thermal behavior are validated together.
